# Entity ***Pools***
+ Layer bronze
+ Priority 1
---
### Imports

In [0]:
%run ../common/bz_constants

In [0]:
%run ../common/medallion_functions

In [0]:
import requests
import time
from pyspark.sql.functions import current_timestamp

### Parameters

In [0]:
layer = dbutils.widgets.get("layer")
entity_name = dbutils.widgets.get("entity_name")
load_pattern = dbutils.widgets.get("load_pattern")
SUBGRAPH_URL = dbutils.widgets.get("SUBGRAPH_URL")

### Variables

In [0]:
pools_list = []
query_variables = {
    "skip": 0
}

In [0]:
pools_query = """
  query PoolsQuery($skip: Int!){
    pools(first: 1000,
      skip: $skip, 
      orderBy: createdAtTimestamp,
      orderDirection: asc
    ) {
      id
      token0 { 
        symbol
        id
        decimals
      }
      token1 { 
        symbol
        id
        decimals
      }
      totalValueLockedToken0
      totalValueLockedToken1
      totalValueLockedETH
      totalValueLockedUSD
      volumeToken0
      volumeToken1
      volumeUSD
      token0Price
      token1Price
      feesUSD
      feeTier
      liquidity
      sqrtPrice
      txCount
      createdAtTimestamp
      createdAtBlockNumber
    }
  }
  """

### Execution

In [0]:
while True:
    print(f"*INFO*: Count of skipped rows: {query_variables["skip"]}.")
    response = requests.post(SUBGRAPH_URL, json={"query": pools_query, "variables": query_variables})
    pools_data = response.json()["data"][entity_name]

    if "errors" in response.json():
            raise Exception(response.json()["errors"])
    
    pools_list.extend(pools_data)

    time.sleep(0.5)

    rows_loaded = len(pools_data)
    print(f"*INFO*: Loaded rows: {rows_loaded}.")

    if rows_loaded < PAGE_SIZE:
        break
           
    query_variables["skip"] += PAGE_SIZE     

In [0]:
bz_pools_df = spark.createDataFrame(pools_list)
bz_pools_df = bz_pools_df.withColumn("load_timestamp", current_timestamp())

### Save & exit

In [0]:
load_result = load_entity(bz_pools_df,
                        entity_name,
                        load_pattern,
                        layer,
                        "old.id = new.id AND old.token0.id = new.token0.id AND old.token1.id = new.token1.id"
                        )

In [0]:
dbutils.notebook.exit(load_result)